# Lab 03: acquire small scientific inputs and explain failures

Automated Workflows in Satellite Image Processing · v1.0 · 15 October 2026

Lab 02 discovered catalogue items and stored one preview. This lab acquires the red
and near-infrared bands needed for NDVI, but stores only a small area of interest.
You design the new database tables completely and then use the resulting archive
to investigate failures.

Use this notebook for predictions, SQL inspection and explanations. Edit
`student_schema.sql` in your editor. Run acquisition and failure cases in a terminal.
The supplied Python is working code, not a new function-repair exercise.

Deliverables: your full schema SQL, your diagnostic SQL, this notebook with evidence
and explanations, and a small final JSON report. Core work: approximately two hours.


## 1. Setup and continuity

Read README.md for installation. From a terminal in this folder, install Rasterio,
NumPy, Matplotlib and Jupyter into your existing environment if needed:

```bash
conda install -c conda-forge "rasterio>=1.4,<2" numpy matplotlib jupyterlab ipykernel
```

We preserve Lab 02's streams, scenes, downloads, runs and stream_scenes tables. You
can copy your database with --from-lab2 on the first init; README.md gives the command.
The default path creates a separate Lab 03 archive and uses captured actual Lab 01
metadata. The original Lab 02 database is never edited by the copy operation.

The fixed example is from June 2025. It is chosen for reproducibility, not presented
as the newest acquisition. Live acquire commands always read remote COG data;
packaged real subsets are used only by offline checks.


In [3]:
from pathlib import Path
import contextlib
import json
import sqlite3
import sys

PROJECT = next(p.resolve() for p in (Path.cwd(), Path.cwd() / 'Lab03', Path.cwd() / 'Labs' / 'Lab03') if (p / 'lab03_cli.py').is_file())
sys.path.insert(0, str(PROJECT))
import check_lab03

def run_checks():
    previous = sys.argv
    sys.argv = ['check_lab03.py']
    try:
        code = check_lab03.main()
    finally:
        sys.argv = previous
    assert code == 0
    return code

print('Project:', PROJECT)

queries = (PROJECT / 'investigation_queries.sql').read_text().split(';')

def inspect_archive(path, query=1):
    with contextlib.closing(sqlite3.connect(f'{path.as_uri()}?mode=ro', uri=True)) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(queries[query])]

Project: C:\Users\micha\AGH\Automated workflows in satellite image processing\automated-workflows-labs\Labs\Lab03


## 2. Design five complete tables (45–60 minutes)

Read SCHEMA_CONTRACT.md. Write every CREATE TABLE statement yourself in
student_schema.sql. The file initially contains only comments. Required column
names let the supplied service use your schema, but there are no partial table
statements to complete.

Before writing SQL, sketch these relationships and explain the expected cardinality:

| Parent | Child | Question |
|---|---|---|
| scenes | assets | Can one scene have more than one band or metadata revision? |
| assets | subset_requests | Can the same band be stored for two AOIs? |
| subset_requests | download_attempts | Can a request fail and later succeed? |
| runs | download_attempts | Can one invocation attempt more than one band? |
| download_attempts | artifacts | Which attempt first registered these exact bytes? |
| artifacts | artifact_locations | Can a stored file disappear without erasing the artifact's identity? |

Also record each request's latest artifact and each attempt's result artifact.
Those artifacts must belong to the same request, not simply be any existing ID.
The contract explains the composite references needed for that condition.

Which facts belong to current state, and which must remain as history? Do not
make run_id or subset_id unique in the attempts table. Predict what that would
prevent during recovery.


In [2]:
run_checks()


attempt=attempt-cd0cdc8b-ce5f-4f4b-8a4a-8de1e65e07ee code=FILE_MISSING Recorded file is absent: real_red_aoi.tif
attempt=attempt-30aa41a7-03f4-4ed1-9909-ce879de6d87c code=FILE_CORRUPT Size/hash mismatch for real_red_aoi.tif


PASS: five complete table definitions and column contract
PASS: primary keys and scene/asset/request/run foreign keys
PASS: AOI identity, uniqueness and allowed states
PASS: attempt outcomes, timestamps and byte counts
PASS: composite relationships prevent cross-AOI artifact references
PASS: real-file verification, missing/corrupt diagnosis and preserved success history
PASS: real B04/B08 grids, AOI masking, scale/offset and pixel cap
PASS: server ignoring Range is refused without reading its body
All checks used actual captured EO data; no live network requests were made.


0

Scene→assets, asset→requests, request/run→attempts and artifact→locations are 1:N; attempts preserve history while request/location status describes current state. UNIQUE(asset_id, aoi_key, recipe_version) and UNIQUE(subset_id, sha256) reject duplicate requests/bytes; scene and composite artifact foreign keys reject unknown scenes and results from another AOI, while unique run/subset IDs on attempts would prevent retries.

## 3. Plan before storing scientific data (10 minutes)

Once checks pass, initialize the new archive and inspect the planned size:

```bash
python lab03_cli.py init --config config/krakow_aoi.json
```

```bash
python lab03_cli.py plan --config config/krakow_aoi.json
```

Open config/krakow_aoi.json. Distinguish discovery_bbox, which selects catalogue
candidates, from aoi_bbox, which limits the stored pixels. Both use west, south,
east, north in EPSG:4326. item_id pins one tile; the AOI must fit entirely inside it.

For each band, report source_shape, subset shape, pixel size, CRS, raw pixel bytes,
scale and offset. The supplied default should produce 227 rows by 219 columns at
10 m resolution, much smaller than the 10,980 by 10,980 source tile.

Use the alternative too_large_aoi.json configuration to investigate a budget failure:

```bash
python lab03_cli.py plan --config config/too_large_aoi.json
```

That command is expected to fail before remote raster data is read. Explain why a
local disk limit and an HTTP transfer limit are different controls. Reading a COG
window fetches whole source blocks; network bytes may exceed stored file bytes.

Both bands: 10980×10980 source → 227×219 subset, 10 m pixels, EPSG:32634, 99426 raw bytes each, scale 0.0001 and offset −0.1; discovery_bbox selects scenes, aoi_bbox limits pixels, and the oversized AOI exceeded the pixel budget, while disk/HTTP limits bound stored files/transferred COG blocks respectively.

## 4. Acquire and repeat (10 minutes plus network time)

```bash
python lab03_cli.py acquire --config config/krakow_aoi.json
```

```bash
python lab03_cli.py acquire --config config/krakow_aoi.json
```

```bash
python lab03_cli.py report --config config/krakow_aoi.json
```

The first run requests two real bands via byte ranges and stores two masked AOI
GeoTIFFs. The second checks their size, SHA-256 and raster metadata and records
reused attempts. Explain why reused is not another transfer and why it is still
worth recording.

Open state/catalogue.sqlite in DB Browser. Inspect the five original tables and
your five new tables. Existing preview downloads remain in downloads; scientific
cutout attempts are in download_attempts and link to subset_requests.

First acquisition: 2097153 network bytes per band, storing 75490 B red and 75702 B nir; the repeat verified both files and recorded reused attempts with 0 transferred bytes, preserving verification history.

### One worked SQL join

Run this in Execute SQL. It follows an attempt to its run, AOI request and source
band. Use it as a starting point for your own diagnostic queries.

```sql
SELECT r.run_id, r.command, a.item_id, a.asset_key,
       s.aoi_key, d.status, d.stage, d.network_bytes,
       d.http_status, d.error_code, d.error_message
FROM download_attempts AS d
JOIN runs AS r ON r.run_id = d.run_id
JOIN subset_requests AS s ON s.subset_id = d.subset_id
JOIN assets AS a ON a.asset_id = s.asset_id
ORDER BY d.started_at, d.attempt_id;
```

Write your own query to show the saved paths, file hashes and latest location
states. Compare HTTP bytes with file sizes. Do not sum bytes after joining a
one-to-many location table unless you account for the duplicated attempt rows.


In [5]:
DB = PROJECT / 'state' / 'catalogue.sqlite'
rows = inspect_archive(DB, 2)
print(json.dumps([row for row in rows if row['attempt_status'] == 'succeeded'], indent=2))

[
  {
    "asset_key": "red",
    "attempt_status": "succeeded",
    "stage": "complete",
    "error_code": null,
    "artifact_id": "59c9437e300a3549f75913bc0b4eeace6430e4be531fe988dded691a43ee71db",
    "sha256": "8363a5d2259d08d542985156aae667a6d37211dad1900f8c836c93f055c60a2a",
    "size_bytes": 75490,
    "local_path": "C:\\Users\\micha\\AGH\\Automated workflows in satellite image processing\\automated-workflows-labs\\Labs\\Lab03\\data\\S2C_34UDA_20250605_0_L2A\\48e7066dc251\\red-59c9437e300a3549.tif",
    "location_status": "available"
  },
  {
    "asset_key": "nir",
    "attempt_status": "succeeded",
    "stage": "complete",
    "error_code": null,
    "artifact_id": "dcfea4a7d5a3e6efc655301bc757ff7fe2477914191628bd3c8b81aabf39292e",
    "sha256": "b8cf04708203ae6ee6a7306f4232b7a69370a397a14a05da1dd41bdb45bf0e49",
    "size_bytes": 75702,
    "local_path": "C:\\Users\\micha\\AGH\\Automated workflows in satellite image processing\\automated-workflows-labs\\Labs\\Lab03\\data\\S2C

## 5. Failure case A: one band succeeded, then the connection failed (20 minutes)

Predict the database and file states before running this:

```bash
python lab03_cli.py case --config config/krakow_aoi.json --name network
```

The command creates cases/network/config.json and its own database/data directory,
then deliberately interrupts the nir reader after actual range responses. A
failure exit code is expected. It does not damage the main archive. The connection
failure is controlled; the scene records and imagery are real.

Open cases/network/state/catalogue.sqlite. Before any recovery:

1. Write SQL to identify the failed run, failed band, failure stage and bytes received.
2. Show which band already has an artifact and which has none.
3. Explain why a failed run does not imply that every operation in it failed. How could an INNER JOIN to artifacts hide the failed attempt?
4. Explain why advancing a discovery watermark would not prove a band was stored.

Record evidence, then recover:

```bash
python lab03_cli.py acquire --config cases/network/config.json
```

Show that the red file was verified/reused, the nir operation was retried, and the
original failure remains in the attempts table. Do not overwrite the failed row
to make the history look successful. Finish with a small before/after table of
run, band, attempt status, stage and artifact identity.

SQL query 2: nir failed at read_window after 262145 B (RANGE_READ_FAILED), while red succeeded; recovery reused red and acquired nir without erasing the failure. LEFT JOIN retains attempts without artifacts, and a discovery watermark tracks metadata rather than stored bands.

| Phase | Run prefix | Band | Status | Stage | HTTP bytes | Artifact prefix |
|---|---|---|---|---|---|---|
| Before | 7b9fa330 | red | succeeded | complete | 2097153 | 59c9437e300a |
| Before | 7b9fa330 | nir | failed | read_window | 262145 | none |
| After | da112268 | red | reused | cache_verify | 0 | 59c9437e300a |
| After | da112268 | nir | succeeded | complete | 2097153 | dcfea4a7d5a3 |

In [8]:
network_rows = inspect_archive(PROJECT / 'cases' / 'network' / 'state' / 'catalogue.sqlite')
for row in network_rows:
    print(row['asset_key'], row['status'], row['stage'], row['network_bytes'], row['error_code'], row['artifact_id'])

[
  {
    "run_id": "lab03-7b9fa330-ca60-4a9a-9cd4-1c96841c0b0d",
    "run_status": "failed",
    "asset_key": "red",
    "status": "succeeded",
    "stage": "complete",
    "network_bytes": 2097153,
    "error_code": null,
    "artifact_id": "59c9437e300a3549f75913bc0b4eeace6430e4be531fe988dded691a43ee71db",
    "sha256": "8363a5d2259d08d542985156aae667a6d37211dad1900f8c836c93f055c60a2a"
  },
  {
    "run_id": "lab03-7b9fa330-ca60-4a9a-9cd4-1c96841c0b0d",
    "run_status": "failed",
    "asset_key": "nir",
    "status": "failed",
    "stage": "read_window",
    "network_bytes": 262145,
    "error_code": "RANGE_READ_FAILED",
    "artifact_id": null,
    "sha256": null
  },
  {
    "run_id": "lab03-da112268-02db-4211-906b-88240edf78ce",
    "run_status": "succeeded",
    "asset_key": "red",
    "status": "reused",
    "stage": "cache_verify",
    "network_bytes": 0,
    "error_code": null,
    "artifact_id": "59c9437e300a3549f75913bc0b4eeace6430e4be531fe988dded691a43ee71db",
    "sha256

## 6. Failure case B: recorded success, missing file (20 minutes)

Predict whether a foreign key can detect a missing disk file. Then run:

```bash
python lab03_cli.py case --config config/krakow_aoi.json --name missing
```

This creates an independent case, acquires real subsets, deletes only that case's
red file, and verifies the archive. A failure exit code is expected. Inspect
cases/missing/state/catalogue.sqlite before recovery.

Write diagnostic SQL and explain:

- Which historical attempt succeeded? Which later operation detected the loss?
- Which table describes the file's current availability, and which describes the
  bytes that previously existed?
- Why should the original successful attempt remain successful in the history?
- What can the database prove, and what required reading the filesystem?

Recover with the same configuration:

```bash
python lab03_cli.py acquire --config cases/missing/config.json
```

Confirm red was restored and nir was reused. If the restored GeoTIFF has the same
hash, explain why it may retain the same artifact identity while having a new
successful transfer attempt.

Optional: --name corrupt truncates a real case file instead of deleting it.
Compare FILE_CORRUPT with FILE_MISSING. --name interrupted stops after writing a
valid temporary cutout but before finalization; explain why no successful artifact
should be registered for that attempt. Generate each case once; later use its
existing configuration to inspect or recover it.

SQL query 3: red's successful acquisition remained in history, but cache verification detected FILE_MISSING and marked its location missing while nir stayed available. Recovery restored red with the same hash/artifact and reused nir; artifacts describe stored bytes, artifact_locations describes current availability, and filesystem checks detect loss that foreign keys cannot.

| Phase | Run prefix | Band | Status | Stage | Artifact prefix | Error |
|---|---|---|---|---|---|---|
| Before | 2540aa52 | red | succeeded | complete | 59c9437e300a | none |
| Before | 2540aa52 | nir | succeeded | complete | dcfea4a7d5a3 | none |
| Before | 52e17055 | red | failed | cache_verify | 59c9437e300a | FILE_MISSING |
| Before | 52e17055 | nir | reused | cache_verify | dcfea4a7d5a3 | none |
| After | b6a76a55 | red | failed | cache_verify | 59c9437e300a | FILE_MISSING |
| After | b6a76a55 | red | succeeded | complete | 59c9437e300a | none |
| After | b6a76a55 | nir | reused | cache_verify | dcfea4a7d5a3 | none |

In [9]:
missing_rows = inspect_archive(PROJECT / 'cases' / 'missing' / 'state' / 'catalogue.sqlite', 2)
for row in missing_rows:
    print(row['asset_key'], row['attempt_status'], row['stage'], row['error_code'], row['location_status'], row['artifact_id'])

[
  {
    "asset_key": "red",
    "attempt_status": "succeeded",
    "stage": "complete",
    "error_code": null,
    "artifact_id": "59c9437e300a3549f75913bc0b4eeace6430e4be531fe988dded691a43ee71db",
    "sha256": "8363a5d2259d08d542985156aae667a6d37211dad1900f8c836c93f055c60a2a",
    "size_bytes": 75490,
    "local_path": "C:\\Users\\micha\\AGH\\Automated workflows in satellite image processing\\automated-workflows-labs\\Labs\\Lab03\\cases\\missing\\data\\S2C_34UDA_20250605_0_L2A\\48e7066dc251\\red-59c9437e300a3549.tif",
    "location_status": "available"
  },
  {
    "asset_key": "nir",
    "attempt_status": "succeeded",
    "stage": "complete",
    "error_code": null,
    "artifact_id": "dcfea4a7d5a3e6efc655301bc757ff7fe2477914191628bd3c8b81aabf39292e",
    "sha256": "b8cf04708203ae6ee6a7306f4232b7a69370a397a14a05da1dd41bdb45bf0e49",
    "size_bytes": 75702,
    "local_path": "C:\\Users\\micha\\AGH\\Automated workflows in satellite image processing\\automated-workflows-labs\\Labs\\

## 7. A new AOI is a different request (10 minutes)

Compare the plan for a smaller AOI of the same scene:

```bash
python lab03_cli.py plan --config config/alternate_aoi.json
```

Optional small acquisition:

```bash
python lab03_cli.py acquire --config config/alternate_aoi.json
```

Write SQL showing that the same source asset can have two subset requests and
that each request points to the correct output. Why would UNIQUE(asset_id) on
subset_requests break this? Why would two independent foreign keys fail to prove
that an artifact belongs to the right request?

The alternate AOI is 114×110 pixels; SQL query 4 on an in-memory database copy shows separate requests for the same asset without downloading more rasters, while UNIQUE(asset_id) would reject the second AOI and separate foreign keys would allow a mismatched artifact/request pair.

In [6]:
import lab03_service as service, lab03_store as store, geo_subset as geo

alternate = service.load_config(PROJECT / 'config' / 'alternate_aoi.json')
with contextlib.closing(sqlite3.connect(':memory:')) as demo, contextlib.closing(sqlite3.connect(f'{DB.as_uri()}?mode=ro', uri=True)) as source:
    source.backup(demo)
    demo.row_factory = sqlite3.Row
    demo.execute('PRAGMA foreign_keys = ON')
    item, plans = service.plan(demo, alternate)
    for plan in plans:
        store.register_request(demo, item, plan['asset_key'], alternate['aoi_bbox'], geo.RECIPE)
    for row in demo.execute(queries[3]):
        print(row['asset_key'], row['aoi_key'][:12], row['subset_id'][:12], row['latest_artifact_id'])

[
  {
    "asset_id": "a2d220b3d0db776bd878281b88dd3d085be4ddc78042674be044110f202bdff8",
    "asset_key": "nir",
    "subset_id": "9d71da0367a3fae0f0f93671894a6c003c4ad3bb3c64b810f29e5e64c56efa49",
    "aoi_key": "02da9b5e2c3e096cd57dbab81d7b4b56f91ccdbde37ec6385f8bd028b265ae54",
    "aoi_json": "{\"bbox\":[19.92,50.045,19.935,50.055],\"crs\":\"EPSG:4326\"}",
    "latest_artifact_id": null,
    "local_path": null,
    "status": null
  },
  {
    "asset_id": "a2d220b3d0db776bd878281b88dd3d085be4ddc78042674be044110f202bdff8",
    "asset_key": "nir",
    "subset_id": "f66f860d150ad4f0a41201bad30d12ab8f99fee60a4d3d25810569c6d44c6f0e",
    "aoi_key": "48e7066dc2510b10454cbc305c90ecb47f71c0966d733d6ab4ec2630c8745724",
    "aoi_json": "{\"bbox\":[19.91,50.04,19.94,50.06],\"crs\":\"EPSG:4326\"}",
    "latest_artifact_id": "dcfea4a7d5a3e6efc655301bc757ff7fe2477914191628bd3c8b81aabf39292e",
    "local_path": "C:\\Users\\micha\\AGH\\Automated workflows in satellite image processing\\automated-wo

## 8. Optional inspection: prepare for NDVI processing

B04/red and B08/nir are the two 10 m inputs for this exercise. Inspect their
reflectance, not unsigned integer arithmetic. Rasterio does not automatically
apply the stored scale and offset when read() returns an array. The offset can
change NDVI; assuming all calibration factors cancel is unsafe.

The cell below reads your main archive only. It does not download anything or
register a derived product. NDVI is shown for inspection; cloud masking, formal
QC and product registration belong to later work.


In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt

paths = {}
if DB.is_file():
    with sqlite3.connect(f'{DB.as_uri()}?mode=ro', uri=True) as conn:
        names = {row[0] for row in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        if 'artifact_locations' in names:
            for band, path in conn.execute("""SELECT a.asset_key,l.local_path FROM subset_requests s
                JOIN assets a USING(asset_id) JOIN artifact_locations l ON l.artifact_id=s.latest_artifact_id
                WHERE l.status='available' ORDER BY s.updated_at"""):
                paths[band] = Path(path)
if set(paths) != {'red','nir'} or not all(p.is_file() for p in paths.values()):
    print('Acquire both main-archive AOI subsets first; no download is performed by this cell.')
else:
    arrays = {}
    grids = {}
    for band, path in paths.items():
        with rasterio.open(path) as src:
            grids[band] = (src.crs, src.transform, src.shape)
            arrays[band] = src.read(1, masked=True).astype('float32') * src.scales[0] + src.offsets[0]
            print(band, src.shape, src.crs, 'scale:', src.scales[0], 'offset:', src.offsets[0])
    assert grids['red'] == grids['nir'], 'Choose the same AOI and aligned bands'
    red, nir = arrays['red'], arrays['nir']
    denominator = nir + red
    with np.errstate(divide='ignore', invalid='ignore'):
        ndvi = np.ma.masked_where(denominator <= 0, (nir-red) / denominator)
    fig, axes = plt.subplots(1,3,figsize=(12,4))
    for ax, values, title in zip(axes,[red,nir,ndvi],['Red reflectance','NIR reflectance','NDVI: inspection only']):
        im = ax.imshow(values, cmap='RdYlGn' if title.startswith('NDVI') else 'gray')
        ax.set_title(title); ax.set_xlabel('Column'); ax.set_ylabel('Row')
        fig.colorbar(im,ax=ax,shrink=0.65)
    plt.tight_layout()
    plt.show()


## Submission and interpretation

Submit student_schema.sql, your investigation_queries.sql, this notebook with
predictions/SQL results/explanations, and a small report:

```bash
python lab03_cli.py report --config config/krakow_aoi.json > lab03_report.json
```

For each required failure, state: symptom, query, observed rows, diagnosis, recovery,
and post-recovery evidence. A screenshot alone is not a diagnosis. Explain which
claims are protected by database constraints and which require checking bytes,
filesystem state or raster metadata.

Keep state/, data/, and cases/ out of Git. Successful offline checks do not claim
that a live request succeeded. If the provider/network is unavailable, retain that
failure evidence and complete the live acquisition later; no synthetic image is
substituted.

Sources: [Rasterio windowed reading/writing](https://rasterio.readthedocs.io/en/latest/topics/windowed-rw.html)
and [Earth Search metadata](https://github.com/Element84/earth-search/blob/main/README.md).
